# 6. Agents and Tools

Let the model choose between lookup and calculation tools.

> 本节展示 Agent 如何理解用户意图、选择工具、执行多步任务，并生成最终回答。


In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from dotenv import load_dotenv

load_dotenv()
MODEL = "openai:gpt-4o-mini"


## 1. Define the tools

A tool needs three things:

1. A clear **name**
2. Accurate **type hints**
3. A precise **docstring**

> 工具名称、参数类型和说明文字会共同影响模型是否选择该工具，以及如何生成参数。


In [ ]:
@tool
def plan_lookup(plan: str) -> str:
    """Look up known features and monthly price for a subscription plan."""
    data = {
        "starter": {
            "price": 29,
            "features": "five users",
        },
        "business": {
            "price": 99,
            "features": "audit logs, API access",
        },
    }
    return str(data.get(plan.lower(), "Unknown plan"))


@tool
def total_cost(monthly_price: float, months: int) -> float:
    """Calculate total subscription cost."""
    return monthly_price * months


## 2. Create the agent

The agent receives the model, available tools, and a system instruction.

> Agent 并不是固定按顺序执行函数，而是根据问题动态决定调用哪个工具、调用几次以及何时结束。


In [ ]:
agent = create_agent(
    model=init_chat_model(MODEL),
    tools=[plan_lookup, total_cost],
    system_prompt=(
        "Use tools for plan facts and calculations. "
        "Do not guess subscription details or arithmetic results."
    ),
)


## 3. Invoke the agent


In [ ]:
question = (
    "What does the Business plan include "
    "and what is the cost for 12 months?"
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question,
            }
        ]
    }
)

final_answer = result["messages"][-1].content
print(final_answer)


### Expected final output

```text
The Business plan includes audit logs and API access.
At $99 per month, the total cost for 12 months is $1,188.
```

The exact wording may vary because the final answer is generated by the model.

> 文字表述可能略有不同，但套餐功能、月费和 12 个月总价应保持一致。


## 4. Inspect the tool-calling trace

The returned `messages` list normally contains the user message, AI tool requests, tool results, and the final AI answer.

> 查看完整消息记录可以证明 Agent 实际调用了工具，而不是直接猜测答案。


In [ ]:
for index, message in enumerate(result["messages"], start=1):
    print(f"\n--- Message {index}: {type(message).__name__} ---")
    print("Content:", message.content)

    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        print("Tool calls:", tool_calls)

    tool_call_id = getattr(message, "tool_call_id", None)
    if tool_call_id:
        print("Tool call ID:", tool_call_id)


### Typical execution trace

```text
User
  → asks for Business plan features and 12-month cost

Agent
  → calls plan_lookup(plan="Business")

plan_lookup
  → returns price=99 and features="audit logs, API access"

Agent
  → calls total_cost(monthly_price=99, months=12)

total_cost
  → returns 1188

Agent
  → writes the final customer-facing answer
```


## 5. Relationship Diagram

```mermaid
flowchart LR
    U[User Question] --> A[LLM Agent]
    A -->|Needs plan facts| P[plan_lookup]
    P -->|Price: 99\nFeatures: audit logs, API access| A
    A -->|Needs arithmetic| C[total_cost]
    C -->|1188| A
    A --> F[Final Answer]

    classDef agent fill:#fce8e6,stroke:#d93025,color:#202124;
    classDef tool fill:#e8f0fe,stroke:#1a73e8,color:#202124;
    classDef io fill:#e6f4ea,stroke:#188038,color:#202124;

    class A agent;
    class P,C tool;
    class U,F io;
```

### Core relationship

```text
Agent = decision maker
Tool = executable capability
Tool result = trusted observation
Final answer = model-generated response based on observations
```

> Agent 负责决策，工具负责执行，工具结果作为可信输入返回给 Agent。


## 6. Think Summary

### How do tool names, type hints, and docstrings affect tool selection?

**Tool name**

A descriptive name such as `plan_lookup` helps the model understand the tool's purpose. A vague name such as `run_task` makes tool selection less reliable.

> 清晰的工具名让模型更容易判断何时调用。

**Type hints**

Type hints define the expected argument schema. For example, `monthly_price: float` and `months: int` guide the model to generate valid structured arguments.

> 类型提示决定工具参数的结构，有助于减少错误参数。

**Docstring**

The docstring describes when and why the tool should be used. It acts like an instruction available to the model during tool selection.

> Docstring 相当于给模型看的工具说明书。

### Key takeaway

```text
Good tool design
= clear name
+ precise description
+ strict argument types
+ predictable output
```

Better tool definitions improve routing accuracy, reduce invalid calls, and make the agent easier to debug.

> 工具定义越清晰，Agent 的路由越准确，也越容易测试和排查。


## 7. Interview Q&A

### Q1. What is the difference between a chain and an agent?

**Answer**

A chain follows a predefined sequence of steps. An agent dynamically decides which tools to call and in what order based on the user's request and intermediate results.

> Chain 的流程预先固定；Agent 会根据问题和中间结果动态选择工具和执行顺序。

---

### Q2. Does the LLM execute the Python function itself?

**Answer**

No. The LLM generates a structured tool call containing the tool name and arguments. The LangChain runtime executes the Python function and sends the result back to the model.

> LLM 只生成工具名和参数，真正执行 Python 函数的是应用运行时。

---

### Q3. Why use tools instead of asking the model to answer directly?

**Answer**

Tools provide deterministic calculations and access to trusted external data. This reduces hallucination and keeps business facts and arithmetic grounded in controlled sources.

> 工具提供确定性计算和可信数据，可降低幻觉并提高答案可靠性。


## 8. How to explain what you built

### 30-second interview answer

I created a LangChain agent with two tools. The first tool retrieves subscription-plan facts, and the second calculates total cost. For a question that requires both plan information and arithmetic, the agent first calls the lookup tool, uses the returned monthly price to call the calculation tool, and then produces a final customer-facing response.

> 我创建了一个包含查询和计算工具的 LangChain Agent。Agent 会先查套餐，再使用返回的月费计算总价，最后生成面向用户的回答。

### One-line résumé version

> Built a LangChain agent that dynamically routed subscription queries across lookup and calculation tools.
